# Lab: Decision Trees

*In this lab, we will build a Decision Tree model using scikit-learn, guided by the [Foundational Methodology for Data Science](../../../05_methodology/). While real-world projects require comprehensive documentation at each stage, this lab focuses on a streamlined, practical approach to demonstrate the core concepts and workflow. Therefore we will settle with the summaries of hypothetical stage reports.*

---

## Stages 1-5: Business Understanding to Data Understanding
This lab notebook builds directly on the findings from the [Multi-Class Classification Lab](./07_multi_class_classification_lab.ipynb), where we aimed to build a predictive model a model to classify species using flower measurements, reducing need for manual inspection. Therefore Stages 1 through 5 (Business Understanding, Analytic Approach, Data Requirements, Data Collection, and Data Understanding) are considered complete. We will proceed directly with preparing the data for our new Decision Tree model.

---

> ### 📝 1–5. Previous Stages Report (Summary)
> 
> * **Solid Multi-Class Foundations:** In the previous Multi-Class Classification lab, we successfully developed robust baseline models (Softmax Regression, One-vs-Rest, One-vs-One) to classify iris species using flower measurements. All models achieved strong accuracy and well-balanced precision/recall across classes, confirming that the features in the Iris dataset are highly predictive of species.
> * **Key Insights:** Exploratory Data Analysis revealed that petal length and petal width provide almost perfect class separability, with the three species nearly linearly separable in feature space. The dataset is balanced, with no missing values and only minor natural outliers. However, an extremely strong correlation (r=0.96) between petal length and petal width was noted, emphasizing the importance of feature engineering and selection.
> * **Business Objective Met (So Far):** Initial analytic goals were achieved: the models enable rapid, automated species identification, offering a practical alternative to manual inspection for botanical institutes.
> * **New Modeling Direction:** With a clean dataset and high-performing baseline models established, the project now transitions to tree-based methods. The goal is to evaluate Decision Trees both as an interpretable classifier and as a foundation for future ensemble methods. We will proceed directly to data preparation and model development.

---

## Stage 6: Data Preparation

As usual, we start with importing the necessary libraries and configuring them.

In [25]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay

# Configurations
plt.style.use("fivethirtyeight")
sns.set_theme(style="white", palette="colorblind")

Let's load our already processed dataset into a DataFrame:

In [26]:
iris_df = pd.read_parquet("../data/processed/iris_multi-class_processed_v1.parquet")
iris_df.sample(5)

,sepal_length,sepal_width,petal_length,petal_width,target,class_name,petal_area,sepal_ratio
109,7.2,3.6,6.1,2.5,2,virginica,15.25,2.000000
24,4.8,3.4,1.9,0.2,0,setosa,0.38,1.411765
88,5.6,3.0,4.1,1.3,1,versicolor,5.33,1.866667
136,6.3,3.4,5.6,2.4,2,virginica,13.44,1.852941
100,6.3,3.3,6.0,2.5,2,virginica,15.00,1.909091


### 6.3. Data Structuring and Final Selection
Now we'll create our feature matrix and target vector. Let's create 3 different feature matrices to experiment with different feature selections.

In [27]:
X_original = iris_df[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
X_engineered = iris_df[["petal_area", "sepal_ratio"]]
X_all = pd.concat([X_original, X_engineered], axis=1)
y = iris_df["target"]

### 6.4. Final Dataset Split

In [28]:
X_train_original, X_test_original, y_train, y_test = train_test_split(X_original, y, test_size=0.2, random_state=42, stratify=y)
X_train_engineered, X_test_engineered, _, _ = train_test_split(X_engineered, y, test_size=0.2, random_state=42, stratify=y)
X_train_all, X_test_all, _, _ = train_test_split(X_all, y, test_size=0.2, random_state=42, stratify=y)


---

> ### 📝 6. Data Preparation Report (Summary)
> 
> - **Dataset Loaded:** The cleaned and processed Iris dataset (with 149 samples and 6 features) was loaded from the prepared analytical file.
> - **Feature Engineering:** In addition to the classic four features (`sepal_length`, `sepal_width`, `petal_length`, `petal_width`), two engineered features were included:
>   - `petal_area` (`petal_length * petal_width`) to capture combined petal size.
>   - `sepal_ratio` (`sepal_length / sepal_width`) to capture sepal shape.
> - **Feature Sets Created:**
>   - **Original Features:** The classic four measurements.
>   - **Engineered Features:** Only `petal_area` and `sepal_ratio`.
>   - **All Features:** Combination of original and engineered features.
> - **Train/Test Split:** Each feature set was split into training and test sets (80/20) using stratified sampling to preserve class balance. The splits were performed with a fixed random seed for reproducibility.
> - **No Scaling Needed:** As decision trees are invariant to feature scaling, no standardization or normalization was applied.

---

## Stage 7: Modeling

### 7.1. Build and Train the Baseline and Candidate Models
Let's use `X_original` for our baseline model, and the other two feature matrices for candidates:

In [29]:
clf_original = DecisionTreeClassifier(random_state=42).fit(X_train_original, y_train)
clf_engineered = DecisionTreeClassifier(random_state=42).fit(X_train_engineered, y_train)
clf_all = DecisionTreeClassifier(random_state=42).fit(X_train_all, y_train)


---

> ### 📝 7. Modeling Report (Summary)
> 
> - **Modeling Approach:** Three separate Decision Tree classifiers were trained to assess the impact of original versus engineered features on multi-class iris classification.
> - **Baseline Model:** Trained using only the four original measurement features (sepal length, sepal width, petal length, petal width).
> - **Candidate Models:**
>   - **Engineered Features Candidate:** Trained using only the two engineered domain features (`petal_area`, `sepal_ratio`).
>   - **All Features Candidate:** Trained using a combined set of both original and engineered features.
> - **Training:** All models were trained with identical random seeds and parameters for fair comparison. The models learned exclusively from the training partition prepared earlier.
> - **Next Steps:** Each model will be evaluated on the held-out test set to compare their predictive performance and assess whether engineered features offer improvements over the original measurements.

---

## Stage 8: Model Evaluation
Now that we have trained our three models, it's time to evaluate their performance on the held-out test set using the metrics defined in the Analytic Approach stage.

In [32]:
# Make predictions on the test sets using all three feature matrices
y_preds_original = clf_original.predict(X_test_original)
y_preds_engineered = clf_engineered.predict(X_test_engineered)
y_preds_all = clf_all.predict(X_test_all)

In [33]:
# Calculate evaluation metrics for each model
def calculate_metrics(y_true, y_preds):
    accuracy = accuracy_score(y_true, y_preds)
    precision = precision_score(y_true, y_preds, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_preds, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_preds, average='weighted', zero_division=0)
    return accuracy, precision, recall, f1

# Compile the results into a DataFrame for comparison
results = {
    'Features': ['Original', 'Engineered', 'All'],
    'Accuracy': [],
    'Precision': [],
    'Recall': [],
    'F1-Score': []
}

for y_pred in [y_preds_original, y_preds_engineered, y_preds_all]:
    accuracy, precision, recall, f1 = calculate_metrics(y_test, y_pred)
    results['Accuracy'].append(accuracy)
    results['Precision'].append(precision)
    results['Recall'].append(recall)
    results['F1-Score'].append(f1)

results_df = pd.DataFrame(results)
print("Model Evaluation Results:")
display(results_df)

Model Evaluation Results:


,Features,Accuracy,Precision,Recall,F1-Score
0,Original,0.933333,0.933333,0.933333,0.933333
1,Engineered,0.900000,0.902357,0.900000,0.899749
2,All,0.900000,0.902357,0.900000,0.899749
